In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
data=pd.read_csv("garment.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'garment.csv'

In [ ]:
data

In [ ]:
df=pd.DataFrame(data)

In [ ]:
df

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
c=df.corr(numeric_only=True)

In [ ]:
c

In [ ]:
sns.heatmap(c,annot=True)

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(c,annot=True,fmt=".2f",linewidth=0.5,cbar=True,square=True)

In [ ]:
rename_dict = {'team': 'TEAM_ID','targeted_productivity': 'TARGET_PROD','smv': 'SMV','wip': 'WIP', 'over_time': 'OVERTIME','incentive': 'INCENTIVE', 'idle_time': 'IDLE_TIME','idle_men': 'IDLE_MEN','no_of_style_change': 'STYLE_CHANGES', 'no_of_workers': 'WORKERS_COUNT','actual_productivity': 'ACTUAL_PROD'
}
df = df.rename(columns=rename_dict)

In [ ]:
df.columns

In [ ]:

replace_dict = { 'team': {1: 'M', 2: 'F', 3: 'OTHERS', 0: 'OTHERS'},
                 'no_of_style_change': {1: 'UG', 2: 'PG', 3: 'HIGH_SCHOOL', 4: 'OTHERS', 0: 'OTHERS',  5: 'OTHERS', 6: 'OTHERS'},
                'actual_productivity': {1: 'YES', 0: 'NO' }}
df.replace(replace_dict, inplace=True)
df

In [ ]:
df['ACTUAL_PROD'].value_counts()

In [ ]:
sns.boxplot(data=df)

In [ ]:
def fix_outliers_iqr(df):
    num=df.select_dtypes(include='number').columns
    for col in num:
        Q1=df[col].quantile(0.25)
        Q3=df[col].quantile(0.75)
        IQR= Q3 - Q1
        lower_bound=  Q1 - 1.5 * IQR
        upper_bound= Q3 + 1.5 * IQR
        df[col]=df[col].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)
    return df    


In [ ]:
df1= fix_outliers_iqr(df)
df1

In [ ]:
sns.boxplot(data=df1)

In [ ]:
df1['ACTUAL_PROD'].value_counts()

In [ ]:
df1.head(10)

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['quarter'] = le.fit_transform(df['quarter'])
df['department'] = le.fit_transform(df['department'])
df

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop=None)
day_encoded = ohe.fit_transform(df[['day']])
cols = ohe.get_feature_names_out(['day'])
df_encoded = pd.DataFrame(day_encoded, columns=cols)
df_encoded

In [ ]:
merge=pd.concat([df1.drop('day',axis=1), df_encoded],axis=1)
merge= pd.DataFrame(merge)
merge

In [ ]:

dept_encoded = ohe.fit_transform(merge[['department']])
cols = ohe.get_feature_names_out(['department'])
df_encode = pd.DataFrame(dept_encoded, columns=cols)
df_encode

In [ ]:
df1 = pd.concat([merge.drop('department', axis=1), df_encode], axis=1)
df1

In [ ]:
smote= SMOTE()

In [ ]:
# Create a binary classification target (1 if met target, 0 if not)
df1['target'] = (df1['ACTUAL_PROD'] >= df1['TARGET_PROD']).astype(int)

# Handle missing values in WIP
df1['WIP'] = df1['WIP'].fillna(0)

# Drop non-numeric and target columns to create feature matrix X
X = df1.drop(columns=['date', 'ACTUAL_PROD', 'target'])
y = df1['target']

# Apply SMOTE
X_smote, y_smote = smote.fit_resample(X, y)

In [ ]:
len(y_smote)

In [ ]:
# Create the balanced DataFrame using the resampled X and y
balanced_df = X_smote.copy()
balanced_df['target'] = y_smote

balanced_df

In [ ]:
len(balanced_df)

In [ ]:
balanced_df['target'] = y_smote
balanced_df=balanced_df.drop('target',axis=1)
balanced_df

In [ ]:
balanced_df['target'] = y_smote
balanced_df

In [ ]:
balanced_df['target'].value_counts()

In [ ]:
from sklearn.feature_selection import SelectKBest,f_classif

In [ ]:
skb=SelectKBest(score_func=f_classif,k=25)

In [ ]:
X.columns

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

X_res = balanced_df.drop(columns=['target'])
y_res = balanced_df['target']

skb = SelectKBest(score_func=f_classif, k=10)

X_new = skb.fit_transform(X_res, y_res)
selected_cols = X_res.columns[skb.get_support()]
df_selected = pd.DataFrame(X_new, columns=selected_cols)
df_selected

In [ ]:
selected_columns=X.columns[skb.get_support()]
selected_columns

In [ ]:
X_selected=X[selected_columns]
X_selected

In [ ]:
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': skb.scores_
}).sort_values(by='Score', ascending=False)

feature_scores

In [ ]:
from sklearn.preprocessing import PowerTransformer
pt=PowerTransformer(method='yeo-johnson')
numeric_cols = balanced_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
df_transformed = pd.DataFrame(
    pt.fit_transform(df1[numeric_cols]),
    columns=numeric_cols
)

In [ ]:
df_transformed.skew()

In [ ]:
from sklearn.preprocessing import StandardScaler
ss=StandardScaler()
X_scaled=ss.fit_transform(X_selected)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X_scaled,y,test_size=0.2,random_state=40)

In [ ]:
print('X_train Shape:',X_train.shape)
print('X_test Shape:',X_test.shape)
print('y_train Shape:',y_train.shape)
print('y_test Shape:',y_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
LR=LogisticRegression()

In [ ]:
LR.fit(X_train,y_train)

In [ ]:
y_pred_lr=LR.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy=accuracy_score(y_test,y_pred_lr)

In [ ]:
print(accuracy)

In [ ]:
from sklearn.metrics import classification_report,f1_score,precision_score,recall_score

In [ ]:
accuracy_lr= accuracy_score(y_test,y_pred_lr)
precision_lr= precision_score(y_test,y_pred_lr)
f1_lr= f1_score(y_test,y_pred_lr)
recall_lr= recall_score(y_test,y_pred_lr)

In [ ]:
print("accuracy of Linear Regression:",accuracy_lr)
print("precision of Linear Regression:",precision_lr)
print("f1_Score of Linear Regression:",f1_lr)
print("recall of Linear Regression:",recall_lr)

In [ ]:
print("Classification Report of Linear Regression:\n", classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
DT=DecisionTreeClassifier()

In [ ]:
DT.fit(X_train, y_train)

In [ ]:
y_pred_dt=DT.predict(X_test)

In [ ]:
accuracy_dt= accuracy_score(y_test,y_pred_dt)
precision_dt= precision_score(y_test,y_pred_dt)
f1_dt= f1_score(y_test,y_pred_dt)
recall_dt= recall_score(y_test,y_pred_dt)

In [ ]:
print("accuracy of Linear Regression:",accuracy_dt)
print("precision of Linear Regression:",precision_dt)
print("f1_Score of Linear Regression:",f1_dt)
print("recall of Linear Regression:",recall_dt)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# Now define your models dictionary
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "AdaBoost": AdaBoostClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100)
}


In [ ]:
scores = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy= accuracy_score(y_test,y_pred)
    precision= precision_score(y_test,y_pred)
    f1= f1_score(y_test,y_pred)
    recall= recall_score(y_test,y_pred)
    scores.append({
        "Model": name,
        "accuracy": round(accuracy, 2),
        "precision": round(precision, 2),
        "f1": round(f1, 2),
        "recall": round(recall, 2)
    })
# Convert to DataFrame for nice display
results_df = pd.DataFrame(scores)
print(results_df)